# Code Interpreter (`RunPython`)

`RunPython` runs Python in a **persistent, srt-confined** Jupyter kernel and surfaces results on two channels:

- **text** — stdout, the last expression's value, tracebacks
- **images** — anything the code *displays* (`plt.show()`, `IPython.display`) comes back **inline as an image**
- **files** — paths you list in `artifacts` are reported by path / type / size (never inlined)

Mental model: **display → *see* it**, **save + `artifacts` → *hand it over***.

Needs the `notebook-exec` extra (`jupyter_client` + `ipykernel`), the `srt` CLI for confinement, and `matplotlib` for the plot below. The LLM cell needs a provider whose tool outputs can carry images — **OpenAI Responses / Anthropic / Gemini** (Chat Completions cannot).

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

from grasp_agents import LLMAgent, RunContext
from grasp_agents import render_events
from grasp_agents.llm_providers.openai_responses import OpenAIResponsesLLM
from grasp_agents.sandbox import local_environment
from grasp_agents.tools.bash import Bash
from grasp_agents.tools.code_interpreter import RunPython, RunPythonInput

load_dotenv()

## A safe, confined kernel

`srt` (Anthropic's sandbox-runtime) confines the kernel's **writes** to `allowed_roots` and gates network egress, while the kernel stays a **native process** — so Apple-Silicon **MPS** / CUDA remain reachable (a Linux VM or container would forfeit the GPU).

In [ ]:
workdir = Path("./ci_workdir").resolve()
workdir.mkdir(exist_ok=True)

env = local_environment(
    allowed_roots=[workdir],
    confinement="srt",  # OS confinement; needs the `srt` CLI on PATH
    # keep matplotlib's font cache inside the writable root:
    env={"MPLCONFIGDIR": str(workdir / ".mpl")},
)
ctx = RunContext[None](state=None, environment=env)

## A persistent, multi-step session

The agent runs **two** RunPython steps in one persistent kernel — the second reuses variables from the first (no redefining), proving state carries across calls — then a **Bash** step in the same workspace. Two figures come back as images, and the file saved in step 2 shows up to Bash.

In [ ]:
agent = LLMAgent[str, str, None](
    name="analyst",
    ctx=ctx,  # the session context is bound at construction
    llm=OpenAIResponsesLLM(model_name="gpt-4o-mini"),
    tools=[RunPython(), Bash()],
    sys_prompt=(
        "You are a Python data assistant with two tools that share one "
        "workspace: RunPython (a live, PERSISTENT Jupyter kernel — variables "
        "and imports carry across calls) and Bash (shell commands). To show a "
        "chart, draw with matplotlib and call `plt.show()`. Take one tool call "
        "per step, in order."
    ),
    env_info=False,
)

async for _ in render_events(
    agent.run_stream(
        "Step 1 (RunPython): define `xs = list(range(11))` and "
        "`ys = [x * x for x in xs]`, then plot ys vs xs and show it.\n"
        "Step 2 (RunPython): do NOT redefine xs or ys — reuse them (the kernel "
        "persists) to compute their running cumulative sum, plot it, show it, "
        "and save that figure to 'cumsum.png'.\n"
        "Step 3 (Bash): run `ls -la` to confirm cumsum.png is in the workspace.\n"
        "Then summarize what you did in one line.",
    ),
):
    pass

## Calling the tool directly (no LLM)

`RunPython` is a plain `BaseTool`. Without an `agent_ctx` each call uses a throwaway kernel. Here the code writes a CSV and we list it in `artifacts` — it comes back as a **reference** (path / type / size), not inlined.

In [ ]:
code = """
import csv
with open("data.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["x", "y"])
    for x in range(5):
        w.writerow([x, x * x])
print("wrote", sum(1 for _ in open("data.csv")), "lines")
"""

result = await RunPython().run(
    RunPythonInput(code=code, artifacts=["data.csv"]),
    ctx=ctx,
)
for part in result:
    print(getattr(part, "text", f"<{type(part).__name__}>"))

## Notes

- **Backends** — `confinement` may be `"none"` (no isolation), `"seatbelt"` (macOS), or `"srt"` (recommended); or run on an E2B remote sandbox via `e2b_environment(..., code_interpreter=True)`. `RunPython` is backend-agnostic.
- **Long jobs** — code runs synchronously and blocks the kernel. Put training loops / big downloads in a background script via `Bash` and poll them; don't run them here.
- **State** — `RunPython` keeps its own persistent kernel per agent loop, separate from the notebook kernel used by `RunCell`.